# GEFCom2014 Wind — Zone 1 processing

Source: [Global Energy Forecasting Competition 2014](http://blog.drhongtao.com/2017/03/gefcom2014-load-forecasting-data.html), wind track, Task 15 (the final/most complete rolling-window release). 10 wind farm zones in Australia; using Zone 1.

Raw columns: `TARGETVAR` (normalized wind power output, 0-1) and NWP wind vector components `U10`,`V10` (10m) / `U100`,`V100` (100m). Only 4 raw features -- derived wind speed, direction and shear from the vectors to clear the feature minimum. These are coordinate transformations of real measurements (not synthetic/calendar), so unlike the BDG2 candidate this stays fully physically grounded.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
import numpy as np
from common import report_candidate

RAW_PATH = "../data/raw/gefcom_wind/task15/Task15_W_Zone1_10/Task15_W_Zone1.csv"
PROCESSED_PATH = "../data/processed/gefcom2014_wind_zone1.csv"

df = pd.read_csv(RAW_PATH)
df["timestamp"] = pd.to_datetime(df["TIMESTAMP"], format="%Y%m%d %H:%M")
df = df.set_index("timestamp").drop(columns=["TIMESTAMP", "ZONEID"])
df = df.rename(columns={"TARGETVAR": "wind_power_norm"})
df.shape

In [ ]:
df["ws10"] = np.sqrt(df.U10**2 + df.V10**2)
df["ws100"] = np.sqrt(df.U100**2 + df.V100**2)
# unit vector components double as sin/cos of direction -- avoids the 0/360 discontinuity a raw bearing would have
df["wd10_sin"] = df.V10 / df.ws10.replace(0, np.nan)
df["wd10_cos"] = df.U10 / df.ws10.replace(0, np.nan)
df["wd100_sin"] = df.V100 / df.ws100.replace(0, np.nan)
df["wd100_cos"] = df.U100 / df.ws100.replace(0, np.nan)
df["shear"] = df.ws100 / df.ws10.replace(0, np.nan)

TARGET = "wind_power_norm"
feature_cols = [c for c in df.columns if c != TARGET]
print(len(feature_cols), feature_cols)
(df.isna().mean() * 100).round(3)

`shear` (`ws100`/`ws10`) blows up when `ws10` is near-calm (~0.5% of hours have `ws10` < 0.5 m/s). Max observed shear is ~12.5 against a typical range of 1-2 -- a real physical extreme (stable boundary layer at low wind), not a data error, but a heavy-tailed feature worth watching during scaling in Phase 1 (consider robust/log scaling rather than plain z-score).

In [ ]:
MAX_GAP_HOURS = 3
before = len(df)
df = df.interpolate(method="time", limit=MAX_GAP_HOURS).dropna(how="any")
print(f"dropped {before - len(df)} rows with unresolved gaps")

In [ ]:
report_candidate(df, TARGET, feature_cols, freq="1h", name="GEFCom2014 Wind Zone1 (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")